# Deepfake Detection using XceptionNet (FaceForensics++ C23)

This notebook trains and evaluates a pretrained **XceptionNet** model for binary classification between **real** and **deepfake** facial images.

We use:
- the **FaceForensics++ C23** dataset
- a pretrained model from **ImageNet**
- transfer learning and fine-tuning
- GPU acceleration when available

Metrics reported:
- accuracy
- confusion matrix
- precision / recall / F1-score


In [35]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)
print("GPU dispo :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))


Device utilisé : cuda
GPU dispo : True
Nom du GPU : NVIDIA GeForce RTX 3050 Laptop GPU


In [36]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())


2.5.1+cu121
True


## 1) Dataset Preparation

We use the dataset **FaceForensics++ C23**, extracted from Kaggle.

Two classes:
- **0 = Original**
- **1 = Deepfake**

Images are already extracted into folders.
We read metadata from CSVs (`Original.csv` and `Deepfakes.csv`) and build a balanced dataset.


In [ ]:
#imports

import pandas as pd
from sklearn.model_selection import train_test_split

from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

BASE_DIR = "./faceforensics_c23"

df_real = pd.read_csv(os.path.join(BASE_DIR, "CSVs", "Original.csv"))
df_fake = pd.read_csv(os.path.join(BASE_DIR, "CSVs", "Deepfakes.csv"))

#print the first line on the 2 dataframes
print(df_real.head())
print(df_fake.head())


     filename     label                    features  \
0  543_f2.jpg  Original  Real face, authentic frame   
1  913_f1.jpg  Original  Real face, authentic frame   
2  189_f1.jpg  Original  Real face, authentic frame   
3  214_f3.jpg  Original  Real face, authentic frame   
4  743_f1.jpg  Original  Real face, authentic frame   

                                            filepath  label_id  
0  /kaggle/input/faceforensics-extracted-dataset-...         0  
1  /kaggle/input/faceforensics-extracted-dataset-...         0  
2  /kaggle/input/faceforensics-extracted-dataset-...         0  
3  /kaggle/input/faceforensics-extracted-dataset-...         0  
4  /kaggle/input/faceforensics-extracted-dataset-...         0  
         filename      label                           features  \
0  112_892_f0.jpg  Deepfakes  Manipulated face, synthetic frame   
1  546_621_f1.jpg  Deepfakes  Manipulated face, synthetic frame   
2  509_525_f4.jpg  Deepfakes  Manipulated face, synthetic frame   
3  151_225_

In [40]:
#we merge the two dataframes into one by concatenate them

df = pd.concat([df_real, df_fake], ignore_index=True)

#we print the number of line for the label column 
print(df["label"].value_counts())


label
Original     5000
Deepfakes    5000
Name: count, dtype: int64


In [ ]:
label_map = {"Original": 0, "Deepfakes": 1} 
df["target"] = df["label"].map(label_map)   

In [ ]:
#we need to build the path to the images by this function
def build_path(row):        
    return os.path.join(    
        BASE_DIR,
        "FF++C32-Frames",
        row["label"],       
        row["filename"]     
    )

df["path"] = df.apply(build_path, axis=1)
print(df.head())


     filename     label                    features  \
0  543_f2.jpg  Original  Real face, authentic frame   
1  913_f1.jpg  Original  Real face, authentic frame   
2  189_f1.jpg  Original  Real face, authentic frame   
3  214_f3.jpg  Original  Real face, authentic frame   
4  743_f1.jpg  Original  Real face, authentic frame   

                                            filepath  label_id  target  \
0  /kaggle/input/faceforensics-extracted-dataset-...         0       0   
1  /kaggle/input/faceforensics-extracted-dataset-...         0       0   
2  /kaggle/input/faceforensics-extracted-dataset-...         0       0   
3  /kaggle/input/faceforensics-extracted-dataset-...         0       0   
4  /kaggle/input/faceforensics-extracted-dataset-...         0       0   

                                                path  
0  ./faceforensics_c23\FF++C32-Frames\Original\54...  
1  ./faceforensics_c23\FF++C32-Frames\Original\91...  
2  ./faceforensics_c23\FF++C32-Frames\Original\18...  
3  .

## 2) Séparation du dataset

Les données sont séparées en :
- `train` (60%)
- `validation` (20%)
- `test` (20%)

Nous utilisons `stratify` pour conserver la proportion Original/Deepfake.


In [43]:
#spliting the datas
trainval_df, test_df = train_test_split(
    df,
    test_size=0.2,  #we use 80% of the data to train and 20% to test
    stratify=df["target"],      #to keep the same ratio of data to train and test
)
train_df, val_df = train_test_split(
    trainval_df,
    test_size=0.2,                 # 20% de trainval → ~16% du total
    stratify=trainval_df["target"],
)

print(len(train_df), len(val_df), len(test_df))



6400 1600 2000


## 2) Transforms and DataLoader

We apply:
- `Resize(299x299)` to match XceptionNet input size
- `RandomHorizontalFlip()` as data augmentation
- `Normalize()` to center pixel values

Using `DataLoader` with:
- `shuffle=True` for training
- `num_workers=4` for parallel loading (faster)


In [ ]:
from torch.utils.data import Dataset    #to create our own dataset
from PIL import Image   #to open images
import torchvision.transforms as T  #transform function
import torch    #to create the tensors

#we put all the images at the same size 128x128, then transform it into tensors, and normalize each channel into -1,0 or 1
transform = T.Compose([
    T.Resize((128, 128)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225])
])

class DeepfakeDataset(Dataset):
    def __init__(self, df, transform=None):     
        self.df = df.reset_index(drop=True)       
        self.transform = transform      

    #renvoie le nombre total d'images dans le dataset pour savoir quand s'arreter
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]     #récupère la ligne en fonction de l'itération
        img = Image.open(row["path"]).convert("RGB")    #ouvre l'image via son chemin et vérifie la présence de 3 canaux RGB
        
        if self.transform:      #on applique la normalisation sur l'image
            img = self.transform(img)
        
        label = torch.tensor(row["target"], dtype=torch.long)   #on crée le tensor, de type entier long, a partir du label 0 et 1
        return img, label   #retourne l'image et son label


## 4) Transformations and DataLoader

Transformations :
- `Resize(299x299)` : size input for Xception
- `RandomHorizontalFlip()` : Data augmentation
- `Normalize()` : values normalization

`num_workers=4` :
- load the images **en parallèle**
- train faster


In [45]:
from torch.utils.data import DataLoader     #importer la classe 


#on crée les 3 datasets pour train et test et on applique le transform aux deux
train_dataset = DeepfakeDataset(train_df, transform)    #input : dataframe + normalisation
val_dataset   = DeepfakeDataset(val_df,  transform)
test_dataset  = DeepfakeDataset(test_df, transform)

#On crée les loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)   #input : 1 dataset, la taille 32 images par batch et on active le mélange
                                                                        #pour un meilleur apprentissage
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)   #pareil mais on active pas le mélange pour l'évaluation



## 3) Model: Pretrained XceptionNet (Transfer Learning)

We use **XceptionNet** from the `timm` library.

Key points:
- Pretrained on **ImageNet**
- State-of-the-art (SOTA) architecture
- We replace the final classification layer to output **2 classes**

This allows the model to reuse its learned visual features and adapt to deepfake detection.


## 4) Optimization Setup

Loss:
- `CrossEntropyLoss` (standard for classification)

Optimizer:
- `AdamW` (improved Adam with correct weight decay behavior)
- Learning rate: `1e-4`

Regularization:
- `weight_decay=1e-4` (L2 regularization to reduce overfitting)

Learning rate scheduler:
- `ReduceLROnPlateau` reduces LR when validation loss stops improving

Early stopping:
- stops training if validation loss does not improve for 3 epochs


## 5) Training Loop

For each epoch:
1. Training phase (forward + backward + optimization)
2. Validation phase (evaluation, no gradient updates)
3. Scheduler adjusts the learning rate
4. Best model checkpoint is saved
5. Early stopping prevents unnecessary training


In [49]:
for epoch in range(num_epochs):
    # ----- TRAIN -----
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        _, preds = torch.max(outputs, 1)
        running_correct += (preds == labels).sum().item()
        running_total += labels.size(0)

    train_loss = running_loss / running_total
    train_acc = running_correct / running_total * 100


    # ----- VALIDATION -----
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * labels.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total * 100

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.2f}% "
        f"| Val loss: {val_loss:.4f} | Val acc: {val_acc:.2f}%"
    )


    # ----- SCHEDULER + EARLY STOPPING -----
    scheduler.step(val_loss)

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_model_path)
        print(f"--> Nouveau meilleur modèle sauvegardé (val_loss = {best_val_loss:.4f})")
    else:
        epochs_no_improve += 1
        print(f"Aucune amélioration depuis {epochs_no_improve} époque(s).")
        if epochs_no_improve >= patience:
            print(f"EARLY STOPPING à l'époque {epoch+1}")
            break


Epoch [1/10] Train loss: 0.0399 | Train acc: 98.58% | Val loss: 0.3306 | Val acc: 88.75%
--> Nouveau meilleur modèle sauvegardé (val_loss = 0.3306)
Epoch [2/10] Train loss: 0.0372 | Train acc: 98.53% | Val loss: 0.3000 | Val acc: 89.75%
--> Nouveau meilleur modèle sauvegardé (val_loss = 0.3000)
Epoch [3/10] Train loss: 0.0398 | Train acc: 98.62% | Val loss: 0.3362 | Val acc: 89.88%
Aucune amélioration depuis 1 époque(s).
Epoch [4/10] Train loss: 0.0280 | Train acc: 99.00% | Val loss: 0.3704 | Val acc: 90.00%
Aucune amélioration depuis 2 époque(s).
Epoch [5/10] Train loss: 0.0223 | Train acc: 99.16% | Val loss: 0.3517 | Val acc: 90.31%
Aucune amélioration depuis 3 époque(s).
EARLY STOPPING à l'époque 5


## 6) Evaluation on Test Set

We load the **best model saved** during training and evaluate it on unseen test data.

Metrics reported:
- Test accuracy
- Loss


In [50]:
# Charger le meilleur modèle sauvegardé
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.to(device)
model.eval()        #re switch to evaluation mode

test_loss = 0.0
test_correct = 0
test_total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * labels.size(0)
        _, preds = torch.max(outputs, 1)
        test_correct += (preds == labels).sum().item()
        test_total += labels.size(0)

avg_test_loss = test_loss / test_total
avg_test_acc = test_correct / test_total * 100

print(f"Test loss: {avg_test_loss:.4f} - Test accuracy: {avg_test_acc:.2f}%")


C:\Users\klora\AppData\Local\Temp\ipykernel_39920\2646099867.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path, map_locati

Test loss: 0.3736 - Test accuracy: 88.60%


In [51]:
print(train_df.iloc[0])


filename                                           473_f2.jpg
label                                                Original
features                           Real face, authentic frame
filepath    /kaggle/input/faceforensics-extracted-dataset-...
label_id                                                    0
target                                                      0
path        ./faceforensics_c23\FF++C32-Frames\Original\47...
Name: 4523, dtype: object


## 7) Performance Metrics

To fully analyze performance, we compute:
- Confusion matrix
- Precision
- Recall
- F1-score

These metrics provide more insight than accuracy alone:
- **False positives** (real → deepfake)
- **False negatives** (deepfake → real)


In [52]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

all_labels = []
all_preds = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)

cm = confusion_matrix(all_labels, all_preds)
print("Matrice de confusion :")
print(cm)

print("\nRapport de classification :")
print(classification_report(all_labels, all_preds, target_names=["Original", "Deepfake"]))


Matrice de confusion :
[[889 111]
 [117 883]]

Rapport de classification :
              precision    recall  f1-score   support

    Original       0.88      0.89      0.89      1000
    Deepfake       0.89      0.88      0.89      1000

    accuracy                           0.89      2000
   macro avg       0.89      0.89      0.89      2000
weighted avg       0.89      0.89      0.89      2000



## 8) Conclusion

- XceptionNet successfully detects deepfakes with ~89% accuracy on unseen test data.
- Strength:
  - Good performance on detecting real images (recall ~0.81)
- Limitation:
  - Detecting deepfakes under compression remains challenging
- Results are consistent with findings in the FaceForensics++ research paper.

For future work:
- exploit frequency-domain artifacts
- explore attention-based architectures (Transformers)
- multimodal analysis (video artifacts)
